# Run Matcher 2026 with TextMining24 cases and fullfill document

In [2]:
import sys, os
from pathlib import Path
ROOT_DIR = Path(os.path.dirname(os.path.abspath("__file__"))).resolve().parents[1]
sys.path.append(str(ROOT_DIR))
from benchmark.subcategory_factory import BenchmarkSubcategoryFactory
from benchmark.benchmark_suite import BenchmarkCaseManager, BenchmarkGoetterdammerung

test_suite_manager = BenchmarkCaseManager()
# BenchmarkSubcateroyFactory is optional and fullfill all tests rows for each product column in test_strategy.csv
test_suite_manager.load("test_strategy.csv", BenchmarkSubcategoryFactory())


# A benchmark Group is a set of subgroups that defines a certain subcategory and contains associated tests encapsuled in a BenchmarkCase

# A BenchmarkCase holds a queryset which consists of one or multiple variety strings associated with a certain subcategory.
# every String is one run in API Client BenchmarkGoetterdammerung for Matcher API Götterdämmerung

benchmark_groups = test_suite_manager.get_benchmark_groups()

total = sum(
    len(case.get_queryset())
    for group in benchmark_groups.values()
    for sub_group in group.get_sub_groups().values()
    for case in sub_group.get_cases()
)
print("Total number of test cases: {}".format(total))

# four strategies exists: ["exact_match_strategy", "fuzzy_match_strategy", "vector_match_strategy"] and "oui_match_strategy" see below.
match_strategies = ["exact_match_strategy", "fuzzy_match_strategy", "vector_match_strategy"]
config = {"fuzzy_match_strategy_threshold": 0.9, "vector_match_strategy_threshold": 0.9}
target_system_b = BenchmarkGoetterdammerung(api_key="1234567890abcdef", url=f"http://localhost:5002/api/match_all_with_strategies", match_strategies=match_strategies, config=config)
count, count_all, ground_truth_suggestions, results = target_system_b.run(benchmark_groups)

for category in count.keys():
    print("{} - {} from {} cases succeeded".format(category, count[category], count_all[category]))

print("following suggestions are not matched:")
# these results failed in comparison with ground of truth value. they are listed as suggest
print(ground_truth_suggestions)

test_suite_manager.attach_results(results, count, count_all)

# details fulfill every row and column in the test section with suggestions if Factory is added in the constructor (see above).
# if results are attached to BenchmarkCaseManager saving them on following columns for every product
test_suite_manager.save_details("details_strategy.csv", with_results=False)
# saving statistics (runtime, passed, failed, count all)
test_suite_manager.save_results("results_strategy.csv")

ModuleNotFoundError: No module named 'surroundingkeys'

# Conduct single benchmark cases with text miner 2024 and matcher 2026

In [3]:
import sys, os
from pathlib import Path
ROOT_DIR = Path(os.path.dirname(os.path.abspath("__file__"))).resolve().parents[1]
sys.path.append(str(ROOT_DIR))
from benchmark.benchmark_suite import BenchmarkGoetterdammerung, BenchmarkCase


# Two querys for same product running against Götterdämmerung matcher API.
queryset = ["SIMATIC ET200", "SIMATIC ETT200"]
# Dummy Benchmark Case object: consider ground_truth when you modify queryset
benchmark_case = BenchmarkCase(test_type="Product", group=None, test_name="S7-1500", category="Test – Different Spelling", sub_category="different spelling - acronym", queryset=queryset, ground_truth=['contains("ET200")'], vendor="Siemens")

match_strategies = ["exact_match_strategy", "fuzzy_match_strategy", "vector_match_strategy"]
target_system_a = BenchmarkGoetterdammerung(api_key="1234567890abcdef", url=f"http://localhost:5002/api/match_all_with_strategies", match_strategies=match_strategies)
# running against matcher API with Client method
results = target_system_a.run_single(case=benchmark_case)

for result in results:
    if result.is_match:
        print("succeeded:")
        print(result.suggest)
    else:
        print("failed:")
        print(result.suggest)

# 2nd test with other product
queryset = ["Siemens SIMATIC S7-1500 CPU PLC", "SIMATC S7-1500 CPU PLC"]
benchmark_case = BenchmarkCase(test_type="Product", group=None, test_name="S7-1500", category="Test – Different Spelling", sub_category="different spelling - acronym", queryset=queryset, ground_truth=['contains("SIMATIC S7-1500")'], vendor="Siemens")

match_strategies = ["exact_match_strategy", "fuzzy_match_strategy", "vector_match_strategy"]
config = {"fuzzy_match_strategy_threshold": 0.9  ,"vector_match_strategy_threshold": 0.9}
target_system_a = BenchmarkGoetterdammerung(api_key="1234567890abcdef", url=f"http://localhost:5002/api/match_all_with_strategies", match_strategies=match_strategies, config=config)
results = target_system_a.run_single(case=benchmark_case)

for result in results:
    if result.is_match:
        print("succeeded:")
        print(result.suggest)
    else:
        print("failed:")
        print(result.suggest)


succeeded:
SIMATIC ET200
succeeded:
SIMATIC ET200
succeeded:
SIMATIC S7-1500
succeeded:
SIMATIC S7-1500


# Conduct single query

In [2]:
import sys, os
from pathlib import Path
ROOT_DIR = Path(os.path.dirname(os.path.abspath("__file__"))).resolve().parents[1]
sys.path.append(str(ROOT_DIR))
from benchmark.benchmark_suite import BenchmarkGoetterdammerung, BenchmarkCase

queryset = ["CPU PLC Siemens und oder SIMATIC S7-1500 CPU PLC"]
benchmark_case = BenchmarkCase(test_type="Product", group=None, test_name="S7-1500", category="Test – Different Spelling", sub_category="different spelling - acronym", queryset=queryset, ground_truth=['contains("SIMATIC S7-1500")'], vendor="Siemens")

match_strategies = ["exact_match_strategy", "fuzzy_match_strategy", "vector_match_strategy"]
config = {"fuzzy_match_strategy_threshold": 0.7  ,"vector_match_strategy_threshold": 0.7}
target_system_a = BenchmarkGoetterdammerung(api_key="1234567890abcdef", url=f"http://localhost:5002/api/match_all_with_strategies", match_strategies=match_strategies)
results = target_system_a.run_single(case=benchmark_case)

for result in results:
    if result.is_match:
        print("succeeded:")
        print(result.suggest)
        # if csaf documents are also matched showing them:
        # you can take a look in resources/CSAF ..
        print(result.csaf_ref)
    else:
        print("failed:")
        print(result.suggest)


succeeded:
SIMATIC S7-1500
[['CSAFPID-0001', 'ICSA-14-073-01'], ['CSAFPID-0001', 'ICSA-14-226-01'], ['CSAFPID-0001', 'ICSA-16-040-02'], ['CSAFPID-0078', 'ICSA-17-129-02'], ['CSAFPID-0079', 'ICSA-17-129-02'], ['CSAFPID-0042', 'ICSA-17-339-01'], ['CSAFPID-0043', 'ICSA-17-339-01'], ['CSAFPID-0015', 'ICSA-18-079-02'], ['CSAFPID-0016', 'ICSA-18-079-02'], ['CSAFPID-0008', 'ICSA-18-226-02'], ['CSAFPID-0009', 'ICSA-18-226-02'], ['CSAFPID-0002', 'ICSA-18-282-05'], ['CSAFPID-0003', 'ICSA-18-282-05'], ['CSAFPID-0002', 'ICSA-18-317-05'], ['CSAFPID-0001', 'ICSA-19-036-04'], ['CSAFPID-0002', 'ICSA-19-036-04'], ['CSAFPID-00012', 'ICSA-19-099-03'], ['CSAFPID-00013', 'ICSA-19-099-03'], ['CSAFPID-0025', 'ICSA-19-099-06'], ['CSAFPID-0026', 'ICSA-19-099-06'], ['CSAFPID-0012', 'ICSA-19-134-09'], ['CSAFPID-0013', 'ICSA-19-134-09'], ['CSAFPID-0003', 'ICSA-19-253-03'], ['CSAFPID-00017', 'ICSA-19-253-03'], ['CSAFPID-0067', 'ICSA-19-283-02'], ['CSAFPID-0068', 'ICSA-19-283-02'], ['CSAFPID-0002', 'ICSA-19-344-06'

# Good example with OUI

In [1]:
import sys, os
from pathlib import Path
ROOT_DIR = Path(os.path.dirname(os.path.abspath("__file__"))).resolve().parents[1]
sys.path.append(str(ROOT_DIR))
from benchmark.benchmark_suite import BenchmarkGoetterdammerung, BenchmarkCase


# Query contains OUI. Take a look in resources "latest_oui_lookup.json" for more examples
queryset = ["CPU PLC 38:4B:24 RF615R"]
benchmark_case = BenchmarkCase(test_type="Product", group=None, test_name="S7-1500", category="Test – Different Spelling", sub_category="different spelling - acronym", queryset=queryset, ground_truth=['contains("RF615R")'], vendor="Siemens")

match_strategies = ["oui_match_strategy", "exact_match_strategy", "fuzzy_match_strategy", "vector_match_strategy"]
config = {"fuzzy_match_strategy_threshold": 0.8  ,"vector_match_strategy_threshold": 0.8}
target_system_a = BenchmarkGoetterdammerung(api_key="1234567890abcdef", url=f"http://localhost:5002/api/match_all_with_strategies", match_strategies=match_strategies)
results = target_system_a.run_single(case=benchmark_case)

for result in results:
    if result.is_match:
        print("succeeded:")
        print(result.suggest)
        if result.csaf_ref:
            print(result.csaf_ref)
    else:
        print("failed:")
        print(result.suggest)


succeeded:
SIMATIC Reader RF615R
[['CSAFPID-00059', 'ICSA-19-253-03'], ['CSAFPID-00067', 'ICSA-19-253-03'], ['CSAFPID-000104', 'ICSA-19-253-03'], ['CSAFPID-0004', 'ICSA-21-159-13'], ['CSAFPID-0005', 'ICSA-21-159-13'], ['CSAFPID-0006', 'ICSA-21-159-13'], ['CSAFPID-0004', 'ICSA-24-256-07'], ['CSAFPID-0005', 'ICSA-24-256-07'], ['CSAFPID-0006', 'ICSA-24-256-07']]


# Conduct benchmark cases with text miner 2024 and matcher 2026 with all threshold possibilities and matching strategy permutations

In [ ]:
import sys, os
from pathlib import Path
ROOT_DIR = Path(os.path.dirname(os.path.abspath("__file__"))).resolve().parents[1]
sys.path.append(str(ROOT_DIR))
from benchmark.subcategory_factory import BenchmarkSubcategoryFactory
from benchmark.benchmark_suite import BenchmarkCaseManager, BenchmarkGoetterdammerung

test_suite_manager = BenchmarkCaseManager()
test_suite_manager.load("test_strategy.csv", BenchmarkSubcategoryFactory())
benchmark_groups = test_suite_manager.get_benchmark_groups()

total = sum(
    len(case.get_queryset())
    for group in benchmark_groups.values()
    for sub_group in group.get_sub_groups().values()
    for case in sub_group.get_cases()
)
print("Total number of test cases: {}".format(total))

# defines all permutations for three match strategies exact, fuzzy and vector
test_strategy_combination = [
    ["exact_match_strategy"],
    ["fuzzy_match_strategy"],
    ["vector_match_strategy"],

    ["exact_match_strategy", "fuzzy_match_strategy"],
    ["exact_match_strategy", "vector_match_strategy"],
    ["fuzzy_match_strategy", "vector_match_strategy"],
    ["vector_match_strategy", "fuzzy_match_strategy"],

    ['exact_match_strategy', 'fuzzy_match_strategy', 'vector_match_strategy'],
    ['exact_match_strategy', 'vector_match_strategy', 'fuzzy_match_strategy'],
    ['fuzzy_match_strategy', 'exact_match_strategy', 'vector_match_strategy'],
    ['fuzzy_match_strategy', 'vector_match_strategy', 'exact_match_strategy'],
    ['vector_match_strategy', 'exact_match_strategy', 'fuzzy_match_strategy'],
    ['vector_match_strategy', 'fuzzy_match_strategy', 'exact_match_strategy']
]

folder = "results"
from pathlib import Path
results_dir = Path(folder)
results_dir.mkdir(exist_ok=True)

for strategy_combination in test_strategy_combination:
    for i in range(1, 11):
        threshold = i / 10

        strategie_name = "_".join([x.split("_")[0] for x in strategy_combination])

        ### uses for every match strategy combination
        config = {"fuzzy_match_strategy_threshold": threshold, "vector_match_strategy_threshold": threshold}
        target_system_b = BenchmarkGoetterdammerung(api_key="1234567890abcdef", url=f"http://localhost:5002/api/match_all_with_strategies", match_strategies=strategy_combination, config=config)
        count, count_all, ground_truth_suggestions, results = target_system_b.run(benchmark_groups)

        for category in count.keys():
            print("{} - {} from {} cases succeeded".format(category, count[category], count_all[category]))

        print("following suggestions are not matched:")
        print(ground_truth_suggestions)

        test_suite_manager.attach_results(results, count, count_all)
        test_suite_manager.save_results(f"{folder}/system_2024_2026_{strategie_name}_{threshold}.csv")